# Лабораторная работа 1. Биграммная языковая модель

Цель работы — реализовать языковую модель из счётчиков: оценить $P(w_t\mid w_{t-1})$, сгенерировать текст, сравнить правдоподобие и перплексию. Для основной части используются только стандартная библиотека Python. Готовые NLP-модели и библиотеки не применяются.

**Корпус:** четыре тома романа Л. Н. Толстого «Война и мир» из открытой коллекции [RusLit](https://github.com/d0rj/RusLit). Корпус загружается в память при запуске и в репозиторий не записывается.

In [1]:
from math import exp, isinf, log
from random import Random
from urllib.parse import quote
from urllib.request import Request, urlopen
import re

class SimpleCounter:

    def __init__(self, iterable=None):
        self._data = {}
        if iterable is not None:
            self.update(iterable)

    def update(self, iterable):
        for item in iterable:
            self.add(item)

    def add(self, item, amount=1):
        self._data[item] = self._data.get(item, 0) + amount

    def __getitem__(self, item):
        return self._data.get(item, 0)

    def __setitem__(self, item, value):
        self._data[item] = value

    def get(self, item, default=0):
        return self._data.get(item, default)

    def items(self):
        return self._data.items()

    def keys(self):
        return self._data.keys()

    def values(self):
        return self._data.values()

    def __iter__(self):
        return iter(self._data)

    def __len__(self):
        return len(self._data)

    def __bool__(self):
        return bool(self._data)

    def most_common(self, n=None):
        ordered = sorted(self._data.items(), key=lambda pair: pair[1], reverse=True)
        return ordered if n is None else ordered[:n]

SEED = 42
rng = Random(SEED)

## 1. Корпус и токенизация

Токен — последовательность русских букв по регулярному выражению `r"[а-яё]+"`. Текст переводится в нижний регистр. Знаки препинания, цифры и фрагменты на других языках отбрасываются: так модель предсказывает только слова, а не смешивает слова со служебными символами. Последние 10 % последовательности токенов откладываются до обучения и используются только для проверки.

In [2]:
BASE_URL = 'https://raw.githubusercontent.com/d0rj/RusLit/main/prose/Tolstoy/'
FILENAMES = [
    'Война и мир. Том 1.txt',
    'Война и мир. Том 2.txt',
    'Война и мир. Том 3.txt',
    'Война и мир. Том 4.txt',
]

def download_utf8(url):
    request = Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with urlopen(request, timeout=60) as response:
        return response.read().decode('utf-8')

urls = [BASE_URL + quote(name) for name in FILENAMES]
corpus_text = '\n'.join(download_utf8(url) for url in urls)
print(f'Загружено символов: {len(corpus_text):,}')
print('Источник:', 'https://github.com/d0rj/RusLit')

Загружено символов: 3,501,795
Источник: https://github.com/d0rj/RusLit


In [3]:
TOKEN_RE = re.compile(r'[а-яё]+')

def tokenize(text):
    return TOKEN_RE.findall(text.lower())

tokens = tokenize(corpus_text)
assert len(tokens) >= 100_000, 'Корпус должен содержать не менее 100 000 слов'

split_at = int(len(tokens) * 0.90)
train_tokens_raw = tokens[:split_at]
test_tokens_raw = tokens[split_at:]

word_counts = SimpleCounter(train_tokens_raw)
print(f'Всего слов в корпусе: {len(tokens):,}')
print(f'Обучающая часть: {len(train_tokens_raw):,} слов (90 %)')
print(f'Отложенная часть: {len(test_tokens_raw):,} слов (10 %)')
print(f'Размер словаря обучающей части: {len(word_counts):,}')
print('\n20 самых частых слов:')
for word, count in word_counts.most_common(20):
    print(f'{word:>15}  {count:>7,}')

Всего слов в корпусе: 502,295
Обучающая часть: 452,065 слов (90 %)
Отложенная часть: 50,230 слов (10 %)
Размер словаря обучающей части: 49,930

20 самых частых слов:
              и   21,417
              в   12,307
             не    8,434
            что    7,866
             он    7,381
             на    6,810
              с    6,334
            как    3,983
            его    3,969
              к    3,567
             то    3,514
              я    3,008
             но    2,663
            она    2,631
            это    2,449
           было    2,441
            все    2,120
             по    2,044
         сказал    1,965
             за    1,946


## 2. Счётчики и вероятности

Для каждого контекста $v$ считаются пары $(v,w)$. Без сглаживания вероятность равна $c(v,w)/c(v)$. При `alpha > 0` применяется сглаживание Лапласа: $(c(v,w)+\alpha)/(c(v)+\alpha V)$. Токен `<unk>` включён в словарь: неизвестные слова из отложенной части отображаются в него, поэтому сглаженная модель остаётся определённой и для слов, отсутствовавших при обучении.

In [4]:
class BigramLanguageModel:
    """Биграммная языковая модель, реализованная только на счётчиках."""

    UNK = '<unk>'

    def __init__(self, seed=42):
        self.rng = Random(seed)
        self.bigram = {}
        self.context_totals = SimpleCounter()
        self.vocabulary = set()
        self.vocab_size = 0

    def fit(self, tokens):
        if len(tokens) < 2:
            raise ValueError('Для обучения нужны хотя бы два токена.')
        self.vocabulary = set(tokens) | {self.UNK}
        self.vocab_size = len(self.vocabulary)
        self.bigram = {}
        self.context_totals = SimpleCounter(tokens[:-1])
        for previous, current in zip(tokens, tokens[1:]):
            if previous not in self.bigram:
                self.bigram[previous] = SimpleCounter()
            self.bigram[previous].add(current)
        return self

    def _known(self, word):
        return word if word in self.vocabulary else self.UNK

    def prob(self, v, w, alpha=0.0):
        """Возвращает P(w | v); alpha=0 означает модель без сглаживания."""
        if alpha < 0:
            raise ValueError('alpha не может быть отрицательным.')
        v, w = self._known(v), self._known(w)
        count_vw = self.bigram.get(v, SimpleCounter())[w]
        count_v = self.context_totals[v]
        if alpha == 0:
            return count_vw / count_v if count_v else 0.0
        return (count_vw + alpha) / (count_v + alpha * self.vocab_size)

    def _distribution(self, v, temperature=1.0):
        """Распределение следующего слова без сглаживания, с температурой."""
        if temperature <= 0:
            raise ValueError('Температура должна быть положительной.')
        v = self._known(v)
        successors = self.bigram.get(v, SimpleCounter())
        total = self.context_totals[v]
        if not successors or total == 0:
            return {}
        scaled = {word: (count / total) ** (1 / temperature)
                  for word, count in successors.items()}
        normalizer = sum(scaled.values())
        return {word: value / normalizer for word, value in scaled.items()}

    def generate(self, start, n_tokens=30, temperature=1.0):
        """Генерирует ровно n_tokens слов, начиная со start."""
        if n_tokens < 1:
            raise ValueError('n_tokens должно быть не меньше 1.')
        result = [self._known(start)]
        while len(result) < n_tokens:
            distribution = self._distribution(result[-1], temperature)
            # Для редкого финального слова без продолжений начинаем новую цепочку.
            if not distribution:
                result[-1] = self.rng.choice(list(self.context_totals))
                distribution = self._distribution(result[-1], temperature)
            words, probabilities = zip(*distribution.items())
            result.append(self.rng.choices(words, weights=probabilities, k=1)[0])
        return result

    def logprob(self, text, alpha=0.0):
        """Сумма log P(w_t | w_{t-1}) для строки или последовательности токенов."""
        sequence = tokenize(text) if isinstance(text, str) else list(text)
        if len(sequence) < 2:
            raise ValueError('Нужно не менее двух токенов.')
        sequence = [self._known(word) for word in sequence]
        total_logprob = 0.0
        for v, w in zip(sequence, sequence[1:]):
            probability = self.prob(v, w, alpha=alpha)
            if probability == 0:
                return float('-inf')
            total_logprob += log(probability)
        return total_logprob

    def perplexity(self, tokens, alpha):
        if len(tokens) < 2:
            raise ValueError('Нужно не менее двух токенов.')
        value = self.logprob(tokens, alpha=alpha)
        return float('inf') if isinf(value) else exp(-value / (len(tokens) - 1))

model = BigramLanguageModel(seed=SEED).fit(train_tokens_raw)
print(f'Контекстов: {len(model.context_totals):,}')
print(f'Разных наблюдённых биграмм: {sum(map(len, model.bigram.values())):,}')
print(f'P(\"и\" | \"что\") = {model.prob("что", "и"):.6f}')

Контекстов: 49,930
Разных наблюдённых биграмм: 268,519
P("и" | "что") = 0.005721


## 3. Генерация

Следующее слово выбирается случайно из распределения строки предыдущего слова. Ниже пять последовательностей по 30 слов от разных стартовых слов. Затем для одного и того же старта сравниваются температуры $T=0.5$, $1$ и $2$: вероятности возводятся в степень $1/T$ и перенормируются.

In [5]:
starts = [word for word, _ in model.context_totals.most_common(5)]
for start in starts:
    generated = model.generate(start, n_tokens=30, temperature=1.0)
    print(f'[{start}]', ' '.join(generated))

[и] и матери взглянула на что вот оно делает тебе господи боже мой сказал он видел почти новое пребывание в эрфурт для которого еще больше чем скрипки заиграли гром победы австрия
[в] в конце доклада за другими интересами и голоса говорившие одни били отвесно сверху глядя на его пьер и я буду иметь какое то время этого сражения все откладываемый вопрос одною
[не] не мог перевести через три года с трепетом кабинет к ней думал ростов а й пехотной дивизией под влиянием не то же отвечала ее голос гость может быть он узнавал
[что] что что то что то связь этого огромного ридикюля это подготовлено бесчисленными так или слишком увлечены и чем дело решенное думал он забавляется тихо отвечала она так как лошади батальонного
[он] он бы поименован был человек который по дорожке заставили его на котором муж невольно поддавался веселому настроению духа войска и всё еще удостоивает быть я ей одной стороны поля сражения


In [6]:
temperature_start = starts[0]
for temperature in (0.5, 1.0, 2.0):
    generated = model.generate(temperature_start, n_tokens=30, temperature=temperature)
    print(f'T = {temperature}:', ' '.join(generated))

T = 0.5: и не мог удержать его и изд г и что то что он был в ней и в этом усатом старом давая чувствовать что то что в комнату и как
T = 1.0: и изд г во всем семейством болконских политический термометр на нее что нибудь необыкновенным сходством с правого или общей улыбке тихона и измученные колеблющиеся люди долго радостно насмешливой радостной и
T = 2.0: и долго уже вмещать в это желательно бы трудясь пошел что велено раздеться и московских магазинов сказал вы работники и соли из петербурга князь сейчас выгнать всех приемах выказывал князь


При $T=0.5$ распределение становится более острым: чаще выбираются самые вероятные продолжения, поэтому текст обычно более шаблонный. При $T=2$ распределение выравнивается, редкие варианты появляются чаще, разнообразие растёт, но локальная связность обычно ухудшается. $T=1$ — исходное распределение счётчиков.

## 4. Правдоподобие и сглаживание

Выбирается предложение из отложенной части, а затем его слова случайно перемешиваются. Без сглаживания достаточно одной отсутствующей биграммы, чтобы вероятность произведения стала нулевой, а $\log P$ — равным $-\infty$. Поэтому сравнить два текста по конечным числам без сглаживания невозможно: обе величины часто равны $-\infty$, хотя число и характер невиданных пар могут различаться.

In [7]:
# Находим предложение, которое целиком начинается в отложенных 10 % текста.
sentence_tokens = [tokenize(part) for part in re.split(r'[.!?]+', corpus_text)]
position = 0
connected_raw = None
for sentence in sentence_tokens:
    if position >= split_at and 10 <= len(sentence) <= 30:
        connected_raw = sentence
        break
    position += len(sentence)

# Это защитный вариант на случай необычной пунктуации источника.
if connected_raw is None:
    connected_raw = test_tokens_raw[:20]

connected = [model._known(word) for word in connected_raw]
shuffled = connected.copy()
rng.shuffle(shuffled)

print('Связный фрагмент:')
print(' '.join(connected))
print('\nТе же слова вперемешку:')
print(' '.join(shuffled))

for alpha in (0.0, 0.1):
    lp_connected = model.logprob(connected, alpha=alpha)
    lp_shuffled = model.logprob(shuffled, alpha=alpha)
    print(f'\nalpha={alpha}: log P(связный) = {lp_connected:.3f}; '
          f'log P(перемешанный) = {lp_shuffled:.3f}')

Связный фрагмент:
ты не можешь себе представить как я бываю несчастна <unk> когда ты такой

Те же слова вперемешку:
бываю я можешь <unk> такой как ты несчастна себе представить ты не когда

alpha=0.0: log P(связный) = -inf; log P(перемешанный) = -inf

alpha=0.1: log P(связный) = -94.471; log P(перемешанный) = -111.433


## 5. Оценка на отложенной части

Перплексия считается только на последних 10 % токенов, которые модель не видела при `fit`: $\mathrm{PPL}=\exp\left(-\frac{1}{N}\sum_t\log P(w_t\mid w_{t-1})\right)$. Чем она меньше, тем лучше модель предсказывает этот тестовый текст.

In [8]:
test_tokens = [model._known(word) for word in test_tokens_raw]
results = []
for alpha in (1.0, 0.1, 0.01):
    log_probability = model.logprob(test_tokens, alpha=alpha)
    ppl = model.perplexity(test_tokens, alpha=alpha)
    results.append((alpha, log_probability, ppl))

print(f'{"alpha":>8} | {"log P(test)":>18} | {"perplexity":>12}')
print('-' * 46)
for alpha, log_probability, ppl in results:
    print(f'{alpha:8.2g} | {log_probability:18.2f} | {ppl:12.3f}')

best_alpha, _, best_ppl = min(results, key=lambda row: row[2])
print(f'\nМинимальная перплексия: {best_ppl:.3f} при alpha = {best_alpha}.')

   alpha |        log P(test) |   perplexity
----------------------------------------------
       1 |         -498604.02 |    20467.971
     0.1 |         -466368.73 |    10773.558
    0.01 |         -448033.34 |     7478.698

Минимальная перплексия: 7478.698 при alpha = 0.01.


Слишком большое сглаживание забирает вероятность у часто встречающихся, хорошо подтверждённых данных вариантов и раздаёт её редким или вообще не наблюдавшимся вариантам. В итоге модель меньше доверяет данным.